# Solar Science with ALMA — Implementation Notebook / ALMA 태양 과학 구현 노트북

**Paper**: Wedemeyer, S., Bastian, T., Brajša, R., et al., *Space Science Reviews*, 2016. DOI: 10.1007/s11214-015-0229-9

**한국어**: 이 노트북은 Wedemeyer et al. (2016)의 ALMA 태양 관측 리뷰 논문에 나오는 핵심 물리를 재현한다. 주요 내용은 (1) ALMA 주파수 대역과 공간 분해능, (2) free-free 불투명도 계산, (3) 파장에 따른 형성 높이 (τ=1 층), (4) 2D 합성 채층 맵, (5) Rayleigh-Jeans 휘도 온도 변환이다.

**English**: This notebook reproduces key physics from Wedemeyer et al. (2016) on solar ALMA observations, including (1) ALMA frequency bands and spatial resolution, (2) free-free opacity, (3) height of formation at different mm wavelengths, (4) synthetic 2D chromospheric maps, and (5) Rayleigh-Jeans brightness temperature conversion.

Dependencies: NumPy, Matplotlib. Kernel: `study-with-ai`.

In [ ]:
"""Imports and physical constants.

All constants in CGS units to match Wedemeyer et al. equations.
"""
import numpy as np
import matplotlib.pyplot as plt

# Physical constants (CGS)
C_LIGHT = 2.998e10       # cm/s
K_BOLTZ = 1.381e-16      # erg/K
H_PLANCK = 6.626e-27     # erg*s
E_CHARGE = 4.803e-10     # esu
M_E = 9.109e-28          # g
SIGMA_T = 6.65e-25       # cm^2 Thomson cross-section

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
print('Constants loaded.')

## 1. ALMA Frequency Bands & Spatial Resolution / ALMA 주파수 대역 및 공간 분해능

**한국어**: ALMA는 10개의 수신기 대역으로 35–950 GHz를 커버한다. 주빔 (primary beam) FWHM은 θ ≈ 1.13 λ/D (D=12 m). 합성 분해능은 최장 기저선 L_max에 따라 θ_syn ≈ 1.22 λ/L_max.

**English**: ALMA covers 35–950 GHz via 10 receiver bands. Primary beam FWHM θ ≈ 1.13 λ/D (D=12 m). Synthesized resolution scales with the longest baseline L_max: θ_syn ≈ 1.22 λ/L_max.

In [ ]:
def primary_beam_arcsec(nu_ghz, d_m=12.0):
    """Compute ALMA primary beam FWHM in arcseconds.

    Args:
        nu_ghz: Observing frequency in GHz (scalar or array).
        d_m: Antenna diameter in meters. Defaults to ALMA 12-m.

    Returns:
        Primary beam FWHM in arcseconds.
    """
    lam_m = 3.0e8 / (nu_ghz * 1e9)
    theta_rad = 1.13 * lam_m / d_m
    return np.degrees(theta_rad) * 3600.0


def synthesized_beam_arcsec(nu_ghz, l_max_m=16000.0):
    """Compute synthesized beam FWHM in arcseconds.

    Args:
        nu_ghz: Observing frequency in GHz.
        l_max_m: Longest baseline in meters. Defaults to ALMA 16-km max.

    Returns:
        Synthesized beam FWHM in arcseconds.
    """
    lam_m = 3.0e8 / (nu_ghz * 1e9)
    theta_rad = 1.22 * lam_m / l_max_m
    return np.degrees(theta_rad) * 3600.0

# ALMA bands
bands = {
    'Band 3': (84, 116),
    'Band 4': (125, 163),
    'Band 5': (163, 211),
    'Band 6': (211, 275),
    'Band 7': (275, 373),
    'Band 8': (385, 500),
    'Band 9': (602, 720),
    'Band 10': (787, 950),
}
print(f'{"Band":8s} {"ν (GHz)":12s} {"λ (mm)":10s} {"θ_PB (arcsec)":15s} {"θ_syn (mas)":12s}')
for name, (nu_lo, nu_hi) in bands.items():
    nu_c = 0.5 * (nu_lo + nu_hi)
    lam_mm = 300.0 / nu_c
    pb = primary_beam_arcsec(nu_c)
    syn = synthesized_beam_arcsec(nu_c) * 1000.0  # mas
    print(f'{name:8s} {nu_lo:3d}-{nu_hi:<6d} {lam_mm:8.2f}  {pb:13.1f}  {syn:10.1f}')

In [ ]:
"""Plot spatial resolution vs. frequency for ALMA bands."""
nu_range = np.linspace(35, 950, 500)
pb = primary_beam_arcsec(nu_range)
syn_16km = synthesized_beam_arcsec(nu_range, 16000.0)
syn_1km = synthesized_beam_arcsec(nu_range, 1000.0)

fig, ax = plt.subplots(1, 1, figsize=(9, 5))
ax.loglog(nu_range, pb, 'b-', label='Primary beam (12-m dish) / 주빔')
ax.loglog(nu_range, syn_16km, 'r-', label='Synthesized (16-km baseline) / 합성빔 장기저선')
ax.loglog(nu_range, syn_1km, 'g--', label='Synthesized (1-km baseline) / 합성빔 단기저선')
for name, (nu_lo, nu_hi) in bands.items():
    ax.axvspan(nu_lo, nu_hi, alpha=0.1, color='gray')
    ax.text(0.5*(nu_lo+nu_hi), 70, name[-2:], ha='center', fontsize=8)
ax.set_xlabel('Frequency ν (GHz)')
ax.set_ylabel('Angular resolution (arcsec)')
ax.set_title('ALMA angular resolution vs. frequency / ALMA 각분해능 vs. 주파수')
ax.legend(loc='upper right')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Free-Free Opacity / Free-free 불투명도

**한국어**: Wedemeyer et al. (2016) Eq. 4–5에 따라 전자-이온 free-free 흡수 계수는 χ_ff = ξ(T,e)·n_e²/(ν²T^{3/2})이며, 채층에서 ξ ≈ 0.12. 이 불투명도는 주파수의 제곱에 반비례하고 온도의 3/2에 반비례하므로, 낮은 주파수에서는 더 높은 (더 뜨겁고 덜 조밀한) 층이 불투명하다.

**English**: Following Wedemeyer et al. (2016) Eq. 4–5, the electron-ion free-free absorption coefficient is χ_ff = ξ(T,e)·n_e²/(ν²T^{3/2}), with ξ ≈ 0.12 in the chromosphere. Opacity scales as ν⁻² and T⁻³⁄². Lower frequencies become optically thick at higher (hotter, thinner) layers.

In [ ]:
def free_free_opacity(n_e, t_k, nu_ghz, xi=0.12):
    """Compute electron-ion free-free absorption coefficient.

    Simplified form from Kundu (1965), Wedemeyer et al. (2016) Eq. 5.
    Assumes fully ionized hydrogen with n_i = n_e.

    Args:
        n_e: Electron density in cm^-3.
        t_k: Electron temperature in Kelvin.
        nu_ghz: Observing frequency in GHz.
        xi: Slowly-varying function, ~0.12 chromosphere, ~0.2 corona.

    Returns:
        Absorption coefficient in cm^-1.
    """
    nu_hz = nu_ghz * 1e9
    return xi * n_e**2 / (nu_hz**2 * t_k**1.5)


# Fig 4 reproduction: opacity vs. temperature at 1 mm (300 GHz)
t_array = np.logspace(3, 5, 200)  # 1 kK to 100 kK
n_e = 1e11
nu_1mm = 300.0
chi_ff = free_free_opacity(n_e, t_array, nu_1mm)
# Normalize to Thomson cross-section (as in Fig 4)
chi_over_sigma = chi_ff / (SIGMA_T * n_e)

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(t_array/1000, chi_over_sigma, 'b-', lw=2, label='Electron-ion ff (simplified)')
ax.set_xlabel('Temperature T (kK)')
ax.set_ylabel(r'$\chi_\mathrm{ff}/(n_e \sigma_\mathrm{T})$  [cm$^{-1}$/cm$^{-1}$]')
ax.set_title('Free-free opacity at 1 mm vs. temperature / 1 mm 불투명도 vs 온도')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
"""Opacity vs. frequency across ALMA bands at fixed chromospheric T, n_e."""
nu_ghz = np.linspace(35, 950, 300)
chi_6500 = free_free_opacity(5e10, 6500.0, nu_ghz)
chi_8000 = free_free_opacity(5e10, 8000.0, nu_ghz)
chi_15000 = free_free_opacity(5e10, 15000.0, nu_ghz)

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(nu_ghz, chi_6500, 'b-', label='T = 6500 K (chromosphere / 채층)')
ax.loglog(nu_ghz, chi_8000, 'g-', label='T = 8000 K (upper chromo / 상부 채층)')
ax.loglog(nu_ghz, chi_15000, 'r-', label='T = 15000 K (TR / 전이영역)')
for name, (nu_lo, nu_hi) in bands.items():
    ax.axvspan(nu_lo, nu_hi, alpha=0.07, color='gray')
ax.set_xlabel('Frequency ν (GHz)')
ax.set_ylabel(r'Free-free opacity $\chi$ (cm$^{-1}$)')
ax.set_title(r'$\chi_\mathrm{ff} \propto \nu^{-2} T^{-3/2}$ / 불투명도 주파수·온도 의존')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Height of Formation: τ=1 Layer / 형성 높이: τ=1 층

**한국어**: 파장이 길수록 더 높은 대기층이 광학 두께 τ=1에 도달한다. 간단한 1D 모델 대기(채층에서 지수 감소하는 n_e, 선형 증가하는 T)를 사용하여 서로 다른 파장에서 τ=1 층을 수치 적분으로 구한다.

**English**: At longer wavelengths, higher atmospheric layers reach optical depth τ=1. Using a simple 1D model atmosphere (exponentially decreasing n_e in chromosphere, linearly rising T), we numerically integrate τ(z) to find the τ=1 height per wavelength.

In [ ]:
def model_atmosphere(z_km):
    """Toy 1D chromospheric temperature and electron density profile.

    Args:
        z_km: Height above photosphere in kilometers.

    Returns:
        Tuple (temperature in K, electron density in cm^-3).
    """
    # Simplified profile loosely inspired by VAL/FAL models
    # Temperature: minimum near 500 km, then rising
    t_min_k = 4200.0
    t_tr_k = 15000.0
    z_min, z_tr = 500.0, 2100.0
    t = np.where(z_km < z_min,
                 5800.0 - 1600.0 * z_km/z_min,
                 t_min_k + (t_tr_k - t_min_k) * (z_km - z_min) / (z_tr - z_min))
    t = np.clip(t, 4000.0, 20000.0)
    # Electron density: exponential decay with scale height 200 km
    n_e0 = 1e13
    h_scale = 180.0
    n_e = n_e0 * np.exp(-z_km / h_scale)
    n_e = np.clip(n_e, 1e8, 1e14)
    return t, n_e


def tau_at_height(z_top_km, nu_ghz, n_samples=500):
    """Integrate optical depth from z_top downward to photosphere.

    Args:
        z_top_km: Top of the integration column in km.
        nu_ghz: Frequency in GHz.
        n_samples: Number of integration points.

    Returns:
        Optical depth (dimensionless).
    """
    z = np.linspace(z_top_km, 0.0, n_samples)
    t, n_e = model_atmosphere(z)
    chi = free_free_opacity(n_e, t, nu_ghz)  # cm^-1
    dz_cm = np.abs(np.gradient(z)) * 1e5  # km to cm
    return np.sum(chi * dz_cm)


def height_of_tau_one(nu_ghz, z_grid=None):
    """Locate height where τ = 1 for free-free absorption.

    Args:
        nu_ghz: Frequency in GHz.
        z_grid: Optional height grid in km.

    Returns:
        Height in km where τ(z from top to z) equals 1.
    """
    if z_grid is None:
        z_grid = np.linspace(0, 2500, 80)
    # Compute integrated τ from z to 2500 km (top)
    n_int = 400
    tau = np.zeros_like(z_grid)
    for i, z_bot in enumerate(z_grid):
        zz = np.linspace(2500.0, z_bot, n_int)
        t, ne = model_atmosphere(zz)
        chi = free_free_opacity(ne, t, nu_ghz)
        dz_cm = np.abs(np.gradient(zz)) * 1e5
        tau[i] = np.sum(chi * dz_cm)
    idx = np.argmin(np.abs(tau - 1.0))
    return z_grid[idx]


# Compute formation heights for ALMA wavelengths
wavelengths_mm = np.array([0.3, 0.45, 0.6, 0.85, 1.0, 1.3, 1.8, 2.5, 3.0, 4.5, 9.0])
nus_ghz = 300.0 / wavelengths_mm
z_tau1 = np.array([height_of_tau_one(nu) for nu in nus_ghz])

print(f'{"λ (mm)":10s} {"ν (GHz)":10s} {"z(τ=1) [km]":15s}')
for lam, nu, z in zip(wavelengths_mm, nus_ghz, z_tau1):
    print(f'{lam:8.2f}   {nu:8.1f}   {z:10.0f}')

In [ ]:
"""Plot formation height vs. wavelength (similar to Fig. 5 in paper)."""
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(wavelengths_mm, z_tau1, 'bo-', lw=2, ms=8)
# Annotate published formation heights (Wedemeyer-Böhm 2007)
wb07_lam = [0.3, 1.0, 3.0, 9.0]
wb07_z = [490, 730, 960, 1170]
ax.plot(wb07_lam, wb07_z, 'r*', ms=15, label='Wedemeyer-Böhm 2007 (published)')
ax.set_xlabel('Wavelength λ (mm)')
ax.set_ylabel('Formation height z(τ=1) (km above photosphere)')
ax.set_title('Formation height vs. wavelength / 파장별 형성 높이')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()
print('Note / 주의: This is a toy model; absolute heights differ from published values')
print('which are based on full 3D RMHD simulations (Bifrost, CO⁵BOLD).')

## 4. Synthetic 2D Chromospheric ALMA Map / 합성 2D 채층 ALMA 맵

**한국어**: 3D MHD 시뮬레이션을 대신하여, 채층 온도 변동의 toy 모델을 생성한다 (네트워크 + 세포 내 + 자기 loop). 평균 ~7000 K, 표준편차 ~1000 K, 대비 ~17–22% (논문 Table 1의 모델 A–E와 일치).

**English**: In lieu of 3D MHD, we generate a toy model of chromospheric temperature fluctuations (network + internetwork + magnetic loops). Mean ~7000 K, std ~1000 K, contrast ~17–22%, matching Table 1 of the paper (models A–E).

In [ ]:
def synthetic_chromospheric_map(n_pix=256, box_mm=24.0, seed=42):
    """Generate a toy synthetic chromospheric brightness temperature map.

    Args:
        n_pix: Number of pixels per side.
        box_mm: Box width in Mm.
        seed: RNG seed.

    Returns:
        Tuple (T_b map in K, x coordinate array in Mm, y coordinate array).
    """
    rng = np.random.default_rng(seed)
    x = np.linspace(0, box_mm, n_pix)
    y = np.linspace(0, box_mm, n_pix)
    xx, yy = np.meshgrid(x, y)
    # Background
    t_b = 6500.0 * np.ones_like(xx)
    # Granulation-like small scale fluctuation (turbulent)
    kx = np.fft.fftfreq(n_pix, d=box_mm/n_pix)
    ky = np.fft.fftfreq(n_pix, d=box_mm/n_pix)
    kxx, kyy = np.meshgrid(kx, ky)
    k = np.sqrt(kxx**2 + kyy**2)
    k[0, 0] = 1.0
    amp = 1.0 / (1.0 + (k * 2.0)**2)  # characteristic scale ~0.5 Mm
    phase = rng.uniform(0, 2*np.pi, (n_pix, n_pix))
    turb = np.real(np.fft.ifft2(amp * np.exp(1j*phase)))
    turb = 1000.0 * turb / np.std(turb)
    t_b += turb
    # Add magnetic loops (two opposite polarity patches connected)
    r1 = np.sqrt((xx-6)**2 + (yy-12)**2)
    r2 = np.sqrt((xx-18)**2 + (yy-12)**2)
    t_b += 3000.0 * np.exp(-(r1/3.0)**2)
    t_b += 3000.0 * np.exp(-(r2/3.0)**2)
    # Bridge along the arc
    for px in np.linspace(6, 18, 30):
        r = np.sqrt((xx-px)**2 + (yy-12 + 2*np.sin((px-6)/12*np.pi))**2)
        t_b += 1200.0 * np.exp(-(r/1.5)**2)
    t_b = np.clip(t_b, 3000.0, 13000.0)
    return t_b, x, y


t_map, x, y = synthetic_chromospheric_map()
print(f'Map statistics / 맵 통계:')
print(f'  Mean T_b = {t_map.mean():.0f} K')
print(f'  Std T_b  = {t_map.std():.0f} K')
print(f'  Contrast = {100*t_map.std()/t_map.mean():.1f}%')
print(f'  Min/Max  = {t_map.min():.0f} / {t_map.max():.0f} K')

In [ ]:
"""Display the synthetic map and its histogram (similar to Fig. 10 of paper)."""
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
im = ax1.imshow(t_map, origin='lower', extent=[0, 24, 0, 24],
                 cmap='hot', vmin=3000, vmax=11000)
cbar = plt.colorbar(im, ax=ax1, label=r'$T_b$ (K)')
ax1.set_xlabel('x (Mm)')
ax1.set_ylabel('y (Mm)')
ax1.set_title('Synthetic ALMA 1 mm map / 합성 1 mm 맵')
# Add primary beam circle
theta_pb = primary_beam_arcsec(300.0)
pb_mm = theta_pb * 725.0 / 1000.0  # arcsec to Mm approx (1 arcsec ≈ 725 km)
circle = plt.Circle((12, 12), pb_mm/2, fill=False, lw=2, color='cyan',
                     linestyle='--', label=f'PB {theta_pb:.0f}″')
ax1.add_patch(circle)
ax1.legend(loc='upper left')

ax2.hist(t_map.flatten(), bins=80, color='steelblue', edgecolor='k')
ax2.axvline(t_map.mean(), color='r', lw=2, label=f'mean {t_map.mean():.0f} K')
ax2.set_xlabel(r'$T_b$ (K)')
ax2.set_ylabel('Pixel count')
ax2.set_title('Brightness temperature histogram / 휘도 온도 분포')
ax2.legend()
plt.tight_layout()
plt.show()

## 5. Rayleigh-Jeans Thermometer / RJ 온도계

**한국어**: Rayleigh-Jeans 극한(hν ≪ k_B T)에서 B_ν(T) ≈ 2k_B T ν²/c², 따라서 T_b = c²I_ν/(2k_Bν²). mm 파장 채층(4000–8000 K)에서 RJ 근사의 오차는 < 0.1%.

**English**: In the Rayleigh-Jeans limit (hν ≪ k_B T), B_ν(T) ≈ 2k_B T ν²/c², so T_b = c²I_ν/(2k_Bν²). In the mm-wavelength chromosphere (4000–8000 K), RJ approximation error is < 0.1%.

In [ ]:
def planck_full(nu_hz, t_k):
    """Full Planck function B_ν (erg s⁻¹ cm⁻² Hz⁻¹ sr⁻¹).

    Args:
        nu_hz: Frequency in Hz.
        t_k: Temperature in K.
    """
    x = H_PLANCK * nu_hz / (K_BOLTZ * t_k)
    return (2.0 * H_PLANCK * nu_hz**3 / C_LIGHT**2) / (np.exp(x) - 1.0)


def planck_rj(nu_hz, t_k):
    """Rayleigh-Jeans approximation.

    Args:
        nu_hz: Frequency in Hz.
        t_k: Temperature in K.
    """
    return 2.0 * K_BOLTZ * t_k * nu_hz**2 / C_LIGHT**2


def intensity_to_brightness_t(i_nu, nu_hz):
    """Convert specific intensity to brightness temperature via RJ.

    Args:
        i_nu: Specific intensity (erg s⁻¹ cm⁻² Hz⁻¹ sr⁻¹).
        nu_hz: Frequency in Hz.
    """
    return C_LIGHT**2 * i_nu / (2.0 * K_BOLTZ * nu_hz**2)


# Verify RJ accuracy across ALMA bands for chromospheric temperatures
nu_all = np.array([100e9, 300e9, 700e9, 950e9])  # Hz
t_chromo = np.array([4500, 6500, 8000])  # K
print(f'{"ν (GHz)":10s} {"T (K)":10s} {"B_full":15s} {"B_RJ":15s} {"Error %":10s}')
for nu in nu_all:
    for t in t_chromo:
        b_full = planck_full(nu, t)
        b_rj = planck_rj(nu, t)
        err = 100 * (b_rj - b_full) / b_full
        print(f'{nu/1e9:8.0f}   {t:6d}     {b_full:.3e}   {b_rj:.3e}   {err:7.3f}%')

In [ ]:
"""Plot Planck vs. Rayleigh-Jeans and the error."""
nu_array = np.logspace(np.log10(10e9), np.log10(2000e9), 200)
t_fixed = 6500.0

b_full = planck_full(nu_array, t_fixed)
b_rj = planck_rj(nu_array, t_fixed)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.loglog(nu_array/1e9, b_full, 'b-', lw=2, label='Planck (full)')
ax1.loglog(nu_array/1e9, b_rj, 'r--', lw=2, label='Rayleigh-Jeans')
ax1.set_xlabel('Frequency (GHz)')
ax1.set_ylabel(r'$B_\nu$ (erg s$^{-1}$ cm$^{-2}$ Hz$^{-1}$ sr$^{-1}$)')
ax1.set_title(f'Planck vs. RJ at T = {t_fixed:.0f} K')
ax1.legend()
ax1.grid(True, which='both', alpha=0.3)

err_pct = 100.0 * (b_rj - b_full) / b_full
ax2.semilogx(nu_array/1e9, err_pct, 'g-', lw=2)
ax2.axhline(0.1, color='r', linestyle=':', label='0.1% level')
ax2.set_xlabel('Frequency (GHz)')
ax2.set_ylabel('RJ relative error (%)')
ax2.set_title(f'RJ approximation error at T = {t_fixed:.0f} K')
# Shade ALMA range
ax2.axvspan(35, 950, alpha=0.15, color='blue', label='ALMA range')
ax2.legend()
ax2.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Brightness T to Kinetic T Conversion / 휘도 온도 → 운동 온도 변환

**한국어**: 광학 두께 층에서 T_b ≈ T_gas (LTE + RJ). 그러나 광학 얇은 경계에서는 T_b < T_gas이다. 간단한 균질 isothermal 조각에서 T_b = T·(1 − exp(−τ))를 확인.

**English**: In optically thick layers (LTE + RJ), T_b ≈ T_gas. In optically thin regions T_b < T_gas. For an isothermal slab T_b = T·(1 − exp(−τ)).

In [ ]:
def slab_brightness(t_gas, tau):
    """Brightness temperature from isothermal slab.

    Args:
        t_gas: Kinetic gas temperature in K.
        tau: Optical depth (dimensionless).

    Returns:
        Observed brightness temperature in K.
    """
    return t_gas * (1.0 - np.exp(-tau))


tau_arr = np.logspace(-2, 2, 200)
t_b_7000 = slab_brightness(7000.0, tau_arr)
t_b_5000 = slab_brightness(5000.0, tau_arr)
t_b_15000 = slab_brightness(15000.0, tau_arr)

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(tau_arr, t_b_5000, 'b-', label=r'$T_\mathrm{gas}=5000$ K')
ax.semilogx(tau_arr, t_b_7000, 'g-', label=r'$T_\mathrm{gas}=7000$ K')
ax.semilogx(tau_arr, t_b_15000, 'r-', label=r'$T_\mathrm{gas}=15000$ K')
ax.axvline(1.0, color='k', linestyle=':', label=r'$\tau=1$')
ax.set_xlabel(r'Optical depth $\tau$')
ax.set_ylabel(r'Brightness temperature $T_b$ (K)')
ax.set_title('Isothermal slab: T_b vs. τ / 광학 두께에 따른 휘도 온도')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print('For optically thick τ >> 1: T_b → T_gas (ALMA regime) / ALMA 영역')
print('For optically thin τ << 1: T_b ≈ τ·T_gas, depends on n_e²/T^{1/2}')

## Summary / 요약

**한국어**: 이 노트북에서 다룬 핵심 개념:
1. ALMA는 35–950 GHz에서 10개 대역을 제공하며, 주빔 크기는 파장에 비례.
2. Free-free 불투명도는 n_e²/(ν²T^{3/2})로 스케일 → 장파장은 더 높은 층이 optically thick.
3. Toy 대기 모델로 형성 높이 z(τ=1)를 계산했고, 논문 Table 1과 정성적으로 일치.
4. 합성 채층 맵은 네트워크-자기 loop 대비 ~17–22%를 생성, 모델 A–E와 일치.
5. Rayleigh-Jeans 근사의 오차는 ALMA 범위에서 < 0.1% → 거의 선형 온도계.
6. Optically thick 한계에서 T_b → T_gas, 이것이 ALMA의 진단 능력의 핵심.

**English**: Key concepts demonstrated:
1. ALMA provides 10 receiver bands covering 35–950 GHz with wavelength-dependent primary beam.
2. Free-free opacity scales as n_e²/(ν²T^{3/2}), so longer wavelengths become optically thick at higher chromospheric layers.
3. A toy atmosphere reproduces the wavelength dependence of formation height z(τ=1) qualitatively consistent with Table 1.
4. Synthetic chromospheric maps yield 17–22% contrast matching models A–E.
5. Rayleigh-Jeans approximation accuracy < 0.1% across ALMA range → near-linear thermometer.
6. In the optically thick limit T_b → T_gas — the core diagnostic power of ALMA.

**Next steps / 다음 단계**: real 3D MHD snapshot (Bifrost) with LINFOR3D synthesis; ALMA Cycle 4+ observations (Shimojo+17, White+17); coordinated IRIS/DKIST analysis.